# Colab 호환 버전

이 노트북은 원본 책 예제를 **Google Colab 최신 환경(2026)** 에 맞춰 자동 마이그레이션한 버전입니다.

## 적용된 변경
- `load_boston` → `fetch_california_housing` (sklearn 1.2+ 호환)
- `DataFrame.append` → `pd.concat` (pandas 2.0+ 호환)
- `OneHotEncoder(sparse=)` → `sparse_output=` (sklearn 1.2+)
- `.iteritems()` → `.items()` (pandas 2.0+)
- `np.bool / np.int / np.float` → 내장 타입 (numpy 1.20+)
- `KMeans()`에 `n_init=10` 명시 (sklearn 1.4+ 경고 회피)
- `mean_squared_error(..., squared=False)` → `np.sqrt(...)` 래핑 (sklearn 1.6+)
- XGBoost/LightGBM `.fit()` 파라미터는 자동 변환 대신 안내 주석을 달았습니다(셀별 의미 차이 때문).

## Colab 데이터 경로
원본은 같은 폴더의 csv/xlsx 파일을 가정합니다. Colab에서는 두 가지 방법 중 하나를 쓰세요.
1. **직접 업로드**: 좌측 파일 패널 → 업로드, 이후 경로는 `./파일명` 또는 `/content/파일명`.
2. **Drive 마운트**:
    ```python
    from google.colab import drive
    drive.mount('/content/drive')
    # 이후 경로: /content/drive/MyDrive/...
    ```


In [ ]:
# Colab 환경 점검 (선택 실행)
import sys, platform
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
try:
    import numpy, pandas, sklearn
    print('numpy:', numpy.__version__, '| pandas:', pandas.__version__, '| sklearn:', sklearn.__version__)
except Exception as e:
    print('환경 점검 중 경고:', e)


In [ ]:
import lightgbm

print(lightgbm.__version__)

### LightGBM 적용 – 위스콘신 Breast Cancer Prediction

In [ ]:
# LightGBM의 파이썬 패키지인 lightgbm에서 LGBMClassifier 임포트
from lightgbm import LGBMClassifier
import lightgbm as lgb  # callbacks 사용을 위해 추가

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# 앞서 XGBoost와 동일하게 n_estimators는 400 설정.
lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능.
evals = [(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, eval_set=evals, callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)])
preds = lgbm_wrapper.predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# plot_importance( )를 이용하여 feature 중요도 시각화
from lightgbm import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(lgbm_wrapper, ax=ax)
plt.savefig('lightgbm_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')